# Causal Entropic Forces

Here's how the algorithm works:
1. From the current state, sample `NUM_ROLLOUTS`
2. Estimate the probability of sampling trajectory i `p(traj_i)` using kernel density estimation
3. Define the volume of trajectory i as `vol_i = 1 / p(traj_i)`
4. Calculate the normalized volume of a trajectory `norm_vol_i = vol_i / \sum_i vol_i`
5. Calculate the causal entropic force `2 * T_C / T_R \mean_i f_i * \log norm_vol_i`, where `f_i` is the initial force of trajectory i and the value `T_C / T_R` governs how much more you want to exploit (acting following the entropic force) vs explore (sampling a trajectory)
6. Use the causal entropic force to update the current state

In [1]:
import numpy as np
import scipy

from scipy import linalg

from matplotlib import pyplot as plt
from matplotlib import patches as patches
from matplotlib import cm

import copy
import time
import datetime

In [2]:
np.random.seed(0)

dtype = np.float32

## Hyperparameters

In [3]:
T_R     = 4e5   # temperature of random agent (K)
T_C     = 2e6   # temperature of causal agent (K)
TAU     = 10.   # simulation time horizon (seconds)
EPSILON = .025  # timesteps (seconds)

In [4]:
NUM_ROLLOUTS = 10_000

In [5]:
NUM_SUBSAMPLES = 3

In [6]:
CHUNK_SIZE = 1000

## Particle-in-a-box environment and dynamics

In [7]:
M     = 1e-21                             # mass (kg)
L     = 400                               # length (meters) 
Q_MIN = np.array([0., 0.  ], dtype=dtype) # minimum box displacements
Q_MAX = np.array([L , L/5.], dtype=dtype) # maximum box displacements

In [8]:
def step_p(p, f):
    p     = f * EPSILON
    p_max = M * np.abs(Q_MAX - Q_MIN) / EPSILON
    new_p = np.sign(p) * np.minimum(p_max, np.abs(p))
    return new_p

In [9]:
def step_q(q, p, new_p):
    new_q = q  +  .5 * EPSILON * (p + new_p) / M
    return new_q

In [10]:
def enforce_elastic_collisions(p, q):
    q  = np.maximum(q, 2 * Q_MIN - q)
    p *= np.sign(q - Q_MIN)
    q  = np.minimum(q, 2 * Q_MAX - q)
    p *= np.sign(Q_MAX - q)
    return p, q

In [11]:
def step_phase_space(p, q, f):
    new_p        = step_p(p, f)
    new_q        = step_q(q, p, new_p)
    new_p, new_q = enforce_elastic_collisions(new_p, new_q)
    return new_p, new_q

## Random rollouts

In [12]:
timesteps = int(TAU / EPSILON)

In [13]:
def generate_fs(num_samples):
    # scale calculation formula from https://math.stackexchange.com/a/1426406
    return np.random.normal( loc   = 0. ,
                             scale = np.sqrt(M * scipy.constants.Boltzmann * T_R) / EPSILON ,
                             size  = (num_samples, 2) ).astype(dtype)

In [14]:
def rollouts(p, q, num_samples):
    """
    paths.shape = (timesteps, num_samples, q_dim)
    """
    ps = np.tile(p[None, :], (num_samples, 1))
    qs = np.tile(q[None, :], (num_samples, 1))

    paths = copy.deepcopy(qs[None, :])

    fs       = generate_fs(num_samples)
    first_fs = copy.deepcopy(fs)
    
    for i in range(timesteps):
        ps, qs = step_phase_space(ps, qs, fs)
        paths  = np.append(paths, qs[None, :], axis=0)
        fs     = generate_fs(num_samples)

    return paths, first_fs

## Causal Entropic Forcing

In [15]:
def subsample(paths):
    # idea and code from Google Gemini to speed up KDE
    sub_indices = np.linspace( 0 ,
                               timesteps - 1 ,
                               NUM_SUBSAMPLES ,
                               dtype=np.int32 ) 
    sub_paths = paths[sub_indices, :, :]
    return sub_paths

In [16]:
# The following code is modified from 
# [1]: https://github.com/scipy/scipy/blob/main/scipy/stats/_kde.py
# [2]: https://github.com/scipy/scipy/blob/main/scipy/stats/_stats.pyx#L744
# [3]: code generated by Google Gemini Flash 3.6 to implement gaussian kde
# with diagonal bandwidth matrix (the idea to use a diagonal kernel was also
# suggested by Google Gemini Flash 3.6)
# 
# [1] has a copyright notice which I have reproduced below
# 
# -------------------------------------------------------------------------------
#
#  Define classes for (uni/multi)-variate kernel density estimation.
#
#  Currently, only Gaussian kernels are implemented.
#
#  Written by: Robert Kern
#
#  Date: 2004-08-09
#
#  Modified: 2005-02-10 by Robert Kern.
#              Contributed to SciPy
#            2005-10-07 by Robert Kern.
#              Some fixes to match the new scipy_core
#
#  Copyright 2004-2005 by Enthought, Inc.
#
# -------------------------------------------------------------------------------

from scipy.special import logsumexp

def diag_gaussian_kde_logpdfs(dataset, subsampling=True, chunk_size=CHUNK_SIZE):
    d, m = dataset.shape
    
    std_per_dim = np.std(dataset, axis=1, keepdims=True)
    scotts_factor = np.power(m, -1. / (d + 4))
    diag_bandwidth = std_per_dim * scotts_factor  # shape (d, 1)
    dataset_ = (dataset / diag_bandwidth).T
    
    log_pdfs = np.zeros(m)
    for i in range(0, m, chunk_size):
        arg = np.sum((dataset_ - dataset_[i : i+chunk_size][:, None]) ** 2., axis=-1)
        log_pdfs[i : i+chunk_size] = logsumexp(-.5 * arg, axis=1)

    return log_pdfs

In [17]:
def log_vol_fracs(dataset, subsampling=True):
    if subsampling:
        dataset = subsample(dataset[1:])
    dataset = dataset.transpose((2, 0, 1)).reshape(-1, NUM_ROLLOUTS)
    log_pdfs = diag_gaussian_kde_logpdfs(dataset)
    log_omega = -log_pdfs
    log_volume_fracs = log_omega - logsumexp(log_omega)
    return log_volume_fracs

In [18]:
def entropic_force(p, q):
    paths, fs = rollouts(p, q, NUM_ROLLOUTS)
    return np.mean( fs * log_vol_fracs(paths)[:, None] , axis=0 )

## Rollouts

In [ ]:
p = np.array([0.   , 0.   ])
q = np.array([L/10., L/10.])

path = copy.deepcopy(q[None, :])

start_time = time.time()
for timestep in range(100):
    elapsed_time = time.time() - start_time
    print(timestep, datetime.timedelta(seconds=int(elapsed_time)), q)
    f_c  = 2 * T_C / T_R * entropic_force(p, q)
    p, q = step_phase_space(p, q, f_c)
    path = np.append(path, q[None, :], axis=0)

0 3.5e-05 [40. 40.]
1 2.75 [41.21628223 39.70610976]
2 5.44 [42.11495477 39.07535546]
3 8.15 [42.95196434 38.81948486]
4 10.8 [43.51721112 38.56901035]
5 13.5 [41.76237385 37.59162437]
6 16.3 [40.54255602 36.3272083 ]
7 19.0 [40.85976988 35.53753444]
8 21.6 [42.08746934 35.62354809]
9 24.3 [42.47186682 36.55299732]
10 27.0 [42.01257069 38.8905018 ]
11 29.7 [41.77340431 41.65500808]
12 32.4 [42.38200982 43.13221715]
13 35.1 [43.74464373 41.7712309 ]
14 37.7 [43.36941142 40.33406033]
15 40.4 [40.88249876 40.31664912]
16 43.2 [39.21170119 39.40658178]
17 45.8 [39.52813831 38.72391436]
18 48.5 [39.86912621 39.33172824]
19 51.2 [39.93990449 39.89404777]
20 53.9 [40.29463228 40.26186265]
21 56.6 [41.60989766 39.4080914 ]
22 59.2 [42.08409339 37.70229641]
23 62.0 [42.02133843 37.02677813]
24 64.6 [42.50054725 34.29871659]
25 67.3 [42.55384411 32.21489154]
26 70.0 [42.2499069  32.36744745]
27 72.7 [41.95244538 31.78290721]
28 75.4 [42.93968961 30.52609787]
29 78.1 [43.10233837 31.31479333]
30 

In [ ]:
fig, ax = plt.subplots(figsize=(13, 13))
ax.scatter(*path[::10].T)
ax.set_xlim(Q_MIN[0], Q_MAX[0])
ax.set_ylim(Q_MIN[1], Q_MAX[1])
ax.set(aspect='equal')
plt.show()